In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../../dataset/news_preprocess.csv")
df = df_selected = df[['Judul', 'Content']]
df

,Judul,Content
0,jokowi kena pakai adat betawi sidang tahun akh...,jakarta kompascom presiden joko widodo pakai b...
1,amnesty international kan indikator krisis dem...,tempoco jakarta amnesty international indonesi...
2,jelang long weekend stasiun kereta cepat halim...,stasiun kereta cepat whoosh halim jakarta timu...
3,kpu tegas pilih tak daftar dpt nyoblos begini ...,jakarta kompascom komisi pilih umum kpu pasti ...
4,kemenag luncur gera senam haji jaga tahan fisi...,tempoco jakarta menteri agama kemenag luncur s...
...,...,...
95,program prioritas kemendikbud guru guru gerak ...,tempoco riau bijak merdeka ajar episode jumlah...
96,cerita korban tipu visa haji ilegal panas ding...,mek kompascom lukman bukan nama benar orang gu...
97,gerindra sebut reshuffle menkumham sinkronisas...,jakarta kompascom ketua hari partai gerindra s...
98,lawrence wong lantik jadi singapura,lawrence wong lantik perdana menteri singapura...


In [4]:
from collections import Counter

def compute_tf(text):
    words = text.split()  # pisahkan kata
    freq = Counter(words)  # hitung jumlah tiap kata

    # tentukan max frequency
    if len(freq) == 0:
        max_freq = 1
    else:
        max_freq = max(freq.values())

    tf = {}

    # hitung TF
    for word in freq:
        count = freq[word]
        tf[word] = count / max_freq

    return tf

In [5]:
df["tf"] = df["Content"].apply(compute_tf)
df

,Judul,Content,tf
0,jokowi kena pakai adat betawi sidang tahun akh...,jakarta kompascom presiden joko widodo pakai b...,"{'jakarta': 0.5, 'kompascom': 0.125, 'presiden..."
1,amnesty international kan indikator krisis dem...,tempoco jakarta amnesty international indonesi...,"{'tempoco': 0.06666666666666667, 'jakarta': 0...."
2,jelang long weekend stasiun kereta cepat halim...,stasiun kereta cepat whoosh halim jakarta timu...,"{'stasiun': 1.0, 'kereta': 0.8, 'cepat': 0.6, ..."
3,kpu tegas pilih tak daftar dpt nyoblos begini ...,jakarta kompascom komisi pilih umum kpu pasti ...,"{'jakarta': 0.07692307692307693, 'kompascom': ..."
4,kemenag luncur gera senam haji jaga tahan fisi...,tempoco jakarta menteri agama kemenag luncur s...,"{'tempoco': 0.04, 'jakarta': 0.08, 'menteri': ..."
...,...,...,...
95,program prioritas kemendikbud guru guru gerak ...,tempoco riau bijak merdeka ajar episode jumlah...,"{'tempoco': 0.0625, 'riau': 0.125, 'bijak': 0...."
96,cerita korban tipu visa haji ilegal panas ding...,mek kompascom lukman bukan nama benar orang gu...,"{'mek': 0.42857142857142855, 'kompascom': 0.14..."
97,gerindra sebut reshuffle menkumham sinkronisas...,jakarta kompascom ketua hari partai gerindra s...,"{'jakarta': 0.6, 'kompascom': 0.2, 'ketua': 0...."
98,lawrence wong lantik jadi singapura,lawrence wong lantik perdana menteri singapura...,"{'lawrence': 0.14285714285714285, 'wong': 1.0,..."


In [6]:
from math import log

def compute_idf(df, column):
    N = len(df)  # jumlah dokumen
    word_doc_count = {}

    # hitung jumlah dokumen yang mengandung tiap kata
    for text in df[column]:
        words = text.split()
        unique_words = set(words)

        for word in unique_words:
            if word in word_doc_count:
                word_doc_count[word] = word_doc_count[word] + 1
            else:
                word_doc_count[word] = 1

    # hitung IDF
    idf = {}
    for word in word_doc_count:
        ni = word_doc_count[word]
        idf[word] = log(N / ni)

    return idf

In [8]:
idf = compute_idf(df, 'Content')
idf

{'baju': 2.995732273553991,
 'ujung': 3.506557897319982,
 'tangguh': 4.605170185988092,
 'sendiri': 1.8971199848858813,
 'luhur': 3.912023005428146,
 'pindah': 2.8134107167600364,
 'indonesia': 0.9675840262617057,
 'cermin': 3.506557897319982,
 'segera': 2.659260036932778,
 'kuat': 1.7719568419318754,
 'kasih': 2.120263536200091,
 'bangsa': 2.120263536200091,
 'kali': 2.0402208285265546,
 'resmi': 1.5606477482646683,
 'zaman': 3.912023005428146,
 'presiden': 1.0216512475319812,
 'bangsawan': 4.605170185988092,
 'representasi': 4.605170185988092,
 'hadap': 1.9661128563728327,
 'tambah': 1.6607312068216509,
 'lama': 1.07880966137193,
 'kena': 1.5606477482646683,
 'negeri': 2.0402208285265546,
 'kompascom': 1.6607312068216509,
 'abetnego': 4.605170185988092,
 'balik': 3.2188758248682006,
 'adapun': 1.6607312068216509,
 'gigih': 3.912023005428146,
 'suku': 3.912023005428146,
 'peci': 4.605170185988092,
 'berani': 2.995732273553991,
 'kata': 0.06187540371808745,
 'kota': 1.6607312068216509,

In [9]:
def compute_tfidf(tf_dict, idf_dict):
    tfidf = {}

    for word in tf_dict:
        if word in idf_dict:
            tfidf[word] = tf_dict[word] * idf_dict[word]
        else:
            tfidf[word] = 0

    return tfidf

In [12]:
df['TF_IDF'] = df['tf'].apply(lambda tf: compute_tfidf(tf, idf))
df

,Judul,Content,tf,TF_IDF
0,jokowi kena pakai adat betawi sidang tahun akh...,jakarta kompascom presiden joko widodo pakai b...,"{'jakarta': 0.5, 'kompascom': 0.125, 'presiden...","{'jakarta': 0.1505525463919608, 'kompascom': 0..."
1,amnesty international kan indikator krisis dem...,tempoco jakarta amnesty international indonesi...,"{'tempoco': 0.06666666666666667, 'jakarta': 0....","{'tempoco': 0.06108604879161034, 'jakarta': 0...."
2,jelang long weekend stasiun kereta cepat halim...,stasiun kereta cepat whoosh halim jakarta timu...,"{'stasiun': 1.0, 'kereta': 0.8, 'cepat': 0.6, ...","{'stasiun': 2.8134107167600364, 'kereta': 2.57..."
3,kpu tegas pilih tak daftar dpt nyoblos begini ...,jakarta kompascom komisi pilih umum kpu pasti ...,"{'jakarta': 0.07692307692307693, 'kompascom': ...","{'jakarta': 0.023161930214147818, 'kompascom':..."
4,kemenag luncur gera senam haji jaga tahan fisi...,tempoco jakarta menteri agama kemenag luncur s...,"{'tempoco': 0.04, 'jakarta': 0.08, 'menteri': ...","{'tempoco': 0.03665162927496621, 'jakarta': 0...."
...,...,...,...,...
95,program prioritas kemendikbud guru guru gerak ...,tempoco riau bijak merdeka ajar episode jumlah...,"{'tempoco': 0.0625, 'riau': 0.125, 'bijak': 0....","{'tempoco': 0.057268170742134694, 'riau': 0.48..."
96,cerita korban tipu visa haji ilegal panas ding...,mek kompascom lukman bukan nama benar orang gu...,"{'mek': 0.42857142857142855, 'kompascom': 0.14...","{'mek': 1.9736443654234679, 'kompascom': 0.237..."
97,gerindra sebut reshuffle menkumham sinkronisas...,jakarta kompascom ketua hari partai gerindra s...,"{'jakarta': 0.6, 'kompascom': 0.2, 'ketua': 0....","{'jakarta': 0.18066305567035296, 'kompascom': ..."
98,lawrence wong lantik jadi singapura,lawrence wong lantik perdana menteri singapura...,"{'lawrence': 0.14285714285714285, 'wong': 1.0,...","{'lawrence': 0.6578814551411559, 'wong': 4.605..."


In [13]:
import math

def cosine_similarity(vec1, vec2):
    # dot product (x . y)
    dot_product = 0
    for word in vec1:
        if word in vec2:
            dot_product += vec1[word] * vec2[word]

    # panjang vektor x
    norm1 = 0
    for value in vec1.values():
        norm1 += value * value
    norm1 = math.sqrt(norm1)

    # panjang vektor y
    norm2 = 0
    for value in vec2.values():
        norm2 += value * value
    norm2 = math.sqrt(norm2)

    # cosine similarity
    if norm1 == 0 or norm2 == 0:
        return 0

    return dot_product / (norm1 * norm2)

In [16]:
similarity_matrix = []

for i in range(len(df)):
    row = []
    for j in range(len(df)):
        sim = cosine_similarity(df['TF_IDF'][i], df['TF_IDF'][j])
        row.append(sim)
    similarity_matrix.append(row)

In [17]:
import pandas as pd

sim_df = pd.DataFrame(similarity_matrix)

In [18]:
sim_df.index = df['Judul']
sim_df.columns = df['Judul']

In [19]:
sim_df

Judul,jokowi kena pakai adat betawi sidang tahun akhir simbol terima kasih jakarta,amnesty international kan indikator krisis demokrasi indonesia,jelang long weekend stasiun kereta cepat halim ramai tumpang malam,kpu tegas pilih tak daftar dpt nyoblos begini mekanisme,kemenag luncur gera senam haji jaga tahan fisik jemaah,bawaslu minta jajar siap lhp hadap gugat sengketa milu,polisi ungkap aktivitas depoy bandar narkoba tewas toren,truk mogok lintas kereta serpong jalan krl ganggu,jalan buntu anies baswedan pilkada jakarta ikut likaliku jalan,warga sebut benda rupa jimat mayat sarung pamulang,...,pkb godok nama maju pilkada jawa timur bukan cak imin,diskon tiket whoosh spesial hut cek info,prabowo batal hadir tutup muktamar pkb cak imin masuk angin,gerindra usung sanujidita pilkada lebak,katakata akhir haniyeh pimpin pergi muncul,program prioritas kemendikbud guru guru gerak hingga pppk,cerita korban tipu visa haji ilegal panas dingin takut tangkap polisi,gerindra sebut reshuffle menkumham sinkronisasi perintah depan,lawrence wong lantik jadi singapura,kpu jakarta gelar rapat pleno agustus bahas status dharma pongrekun
Judul,,,,,,,,,,,,,,,,,,,,,
jokowi kena pakai adat betawi sidang tahun akhir simbol terima kasih jakarta,1.000000,0.023902,0.003675,0.014731,0.010138,0.009185,0.053414,0.001008,0.021046,0.026414,...,0.022441,0.002694,0.022824,0.010828,0.025836,0.019581,0.023556,0.024797,0.011823,0.004709
amnesty international kan indikator krisis demokrasi indonesia,0.023902,1.000000,0.015900,0.028264,0.014115,0.033812,0.012176,0.001869,0.054341,0.015398,...,0.020181,0.009231,0.021418,0.020018,0.032488,0.023068,0.030194,0.047909,0.017733,0.010712
jelang long weekend stasiun kereta cepat halim ramai tumpang malam,0.003675,0.015900,1.000000,0.014937,0.010197,0.005028,0.010769,0.019499,0.004300,0.024215,...,0.025350,0.204780,0.015433,0.004279,0.011617,0.017419,0.010542,0.029410,0.012386,0.018868
kpu tegas pilih tak daftar dpt nyoblos begini mekanisme,0.014731,0.028264,0.014937,1.000000,0.020014,0.067688,0.012906,0.000355,0.047072,0.017126,...,0.051328,0.007198,0.048904,0.006416,0.006619,0.019877,0.009416,0.023325,0.013823,0.065606
kemenag luncur gera senam haji jaga tahan fisik jemaah,0.010138,0.014115,0.010197,0.020014,1.000000,0.010502,0.012100,0.012863,0.010058,0.008981,...,0.003178,0.012568,0.031574,0.003464,0.013945,0.041331,0.159310,0.010406,0.006211,0.005395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
program prioritas kemendikbud guru guru gerak hingga pppk,0.019581,0.023068,0.017419,0.019877,0.041331,0.011133,0.016071,0.000499,0.007408,0.005871,...,0.012866,0.010910,0.020639,0.010185,0.028825,1.000000,0.054641,0.015927,0.012941,0.014712
cerita korban tipu visa haji ilegal panas dingin takut tangkap polisi,0.023556,0.030194,0.010542,0.009416,0.159310,0.012827,0.043768,0.014498,0.009315,0.040902,...,0.018530,0.015606,0.005412,0.004075,0.012730,0.054641,1.000000,0.010445,0.003347,0.004735
gerindra sebut reshuffle menkumham sinkronisasi perintah depan,0.024797,0.047909,0.029410,0.023325,0.010406,0.022878,0.005157,0.001633,0.091574,0.010832,...,0.023175,0.022292,0.054601,0.076040,0.017254,0.015927,0.010445,1.000000,0.030838,0.016983
